# LLM-as-a-Judgeは本当に信頼できるのか -- 評価器を評価する

LLM-as-a-Judgeは、生成AIアプリの品質を自動で採点してくれる便利な仕組みです。しかし「そのジャッジの判定、本当に人間の判断と合っているのか?」を確認しないまま使うと、ズレた採点を信じて誤った意思決定をしてしまいます。

このノートブックでは、ジャッジそのものの信頼性を測る **メタ評価 (evaluating the evaluator)** を、Databricks + MLflow 3で実際に動かします。題材は「ベーカリーのレビュー返信エージェント」の返信が適切かを判定するジャッジです。

やること:
1. 人間がゴールドラベル (pass/fail) を付けた評価データを用意する
2. `make_judge()` でカスタムジャッジを作る
3. ジャッジで全件を採点する
4. ジャッジの判定と人間ラベルの一致度を **一致率** と **Cohen's kappa** で測る (ここが本題)
5. 不一致ケースを掘り下げて「なぜズレたか」を見る
6. (任意) アラインメントでジャッジを人間に寄せて、kappaが上がるかを確認する

一致率だけを見ると高く見えても、クラスの偏りがあると当てにならないため、偶然の一致を割り引くCohen's kappaを併用するのがポイントです。

In [0]:
%pip install --upgrade "mlflow[databricks]>=3.4.0" dspy
dbutils.library.restartPython()

## 1. 設定

ノートブックに紐づくエクスペリメントが自動的に使われるため、`mlflow.set_experiment()` は呼びません。

`JUDGE_MODEL` はDatabricks Foundation Model APIsのエンドポイントを指定します。環境に合わせて変更してください。ジャッジは採点対象の生成に使うモデルとは別系統のモデルを選ぶと、自己評価バイアス (self-enhancement bias) を避けられます。

In [0]:
import mlflow
from typing import Literal
from collections import Counter
import pandas as pd

print(f"MLflow version: {mlflow.__version__}")

# ジャッジに使うモデル (環境に合わせて変更)
# 例: "databricks:/databricks-gpt-oss-120b" / "databricks:/databricks-claude-sonnet-4"
JUDGE_MODEL = "databricks:/databricks-gpt-oss-120b"

# ジャッジ名 (後半のアラインメントで人間フィードバックのnameと一致させる必要があるため定数化)
JUDGE_NAME = "reply_appropriateness"

# 判定ラベル
LABELS = ["pass", "fail"]

## 2. 評価対象データの準備 (人間のゴールドラベル付き)

ベーカリー「小麦の森」のレビュー返信エージェントを想定します。`review` が顧客レビュー、`reply` がエージェントの返信、`human_label` が人間が付けた正解 (pass=適切 / fail=不適切) です。

メタ評価を意味のあるものにするため、判定が割れやすい微妙なケース (存在しないクーポンの捏造、アレルゲンの断定、皮肉の読み違い、中立レビューを絶賛と誤読、など) を意図的に混ぜています。ネガティブ例が約35%になるよう配分しています。

In [0]:
DATA = [
    {
        "id": 1,
        "review": "クロワッサンがサクサクで最高でした!また来ます。",
        "reply": "うれしいお言葉をありがとうございます!またのご来店を心よりお待ちしております。",
        "human_label": "pass",
        "human_note": "ポジティブに対する適切な感謝。",
    },
    {
        "id": 2,
        "review": "楽しみにして行ったのに、15時にはパンがほとんど売り切れ。がっかりしました。",
        "reply": "貴重なご意見ありがとうございます。午後の品揃えが不十分でご不便をおかけし申し訳ございません。焼き上げの時間と数量を見直してまいります。",
        "human_label": "pass",
        "human_note": "謝罪 + 具体的な改善に言及。",
    },
    {
        "id": 3,
        "review": "店員さんの態度が冷たくて不快でした。",
        "reply": "ご来店ありがとうございました!スタッフ一同、またお会いできるのを楽しみにしています。",
        "human_label": "fail",
        "human_note": "苦情を無視し、トーンがずれている。",
    },
    {
        "id": 4,
        "review": "食パンは美味しいけど、正直ちょっと高いかな。",
        "reply": "お褒めいただきありがとうございます!ちなみに毎週火曜は全品20%オフですので、ぜひご利用ください。",
        "human_label": "fail",
        "human_note": "裏取りできない割引を勝手に約束 (ハルシネーション)。ジャッジが見落としやすい。",
    },
    {
        "id": 5,
        "review": "小麦アレルギーがあるのですが、こちらのパンは食べられますか?",
        "reply": "はい、当店のパンはアレルギーの方でも安心してお召し上がりいただけますのでご安心ください!",
        "human_label": "fail",
        "human_note": "根拠のない安全性の断定。健康被害に直結する危険な返信。",
    },
    {
        "id": 6,
        "review": "美味しかったです。",
        "reply": "ありがとうございます!励みになります。またお立ち寄りください。",
        "human_label": "pass",
        "human_note": "短いポジティブに対する妥当な返信。",
    },
    {
        "id": 7,
        "review": "駐車場が分かりにくくて、近くをぐるぐる回ってしまいました。",
        "reply": "ご不便をおかけし申し訳ございません。店舗裏の第2駐車場のご案内看板を増やすなど、分かりやすさの改善を検討いたします。",
        "human_label": "pass",
        "human_note": "指摘点に具体的に対応。",
    },
    {
        "id": 8,
        "review": "パンにビニールの切れ端のようなものが入っていました。",
        "reply": "この度は貴重なご意見をありがとうございます。今後ともよろしくお願いいたします。",
        "human_label": "fail",
        "human_note": "異物混入という重大事案に対し、謝罪も対応もない定型返信。",
    },
    {
        "id": 9,
        "review": "接客も味も文句なし。デートで使えるおしゃれさ。",
        "reply": "ありがとうございます!素敵な時間のお手伝いができて光栄です。またのお越しをお待ちしております。",
        "human_label": "pass",
        "human_note": "ポジティブに沿った返信。",
    },
    {
        "id": 10,
        "review": "レジで待たされた上に、注文と違うパンを渡されました。",
        "reply": "お待たせした上に商品のお渡しにも誤りがあり、大変申し訳ございませんでした。オペレーションを見直し、再発防止に努めます。",
        "human_label": "pass",
        "human_note": "複数の苦情に謝罪 + 対応。",
    },
    {
        "id": 11,
        "review": "まあ普通。可もなく不可もなく。",
        "reply": "この度は最高の体験をご提供できたようで、大変うれしく思います!",
        "human_label": "fail",
        "human_note": "中立レビューを絶賛と誤読。感情の読み違い。ジャッジが割れやすい。",
    },
    {
        "id": 12,
        "review": "子連れでも入りやすくて助かりました。ベビーカーOKなのが嬉しい。",
        "reply": "温かいお言葉ありがとうございます!小さなお子様連れでも安心してお過ごしいただけるよう、これからも努めてまいります。",
        "human_label": "pass",
        "human_note": "指摘に沿った返信。",
    },
    {
        "id": 13,
        "review": "値段の割に量が少ない気がします。",
        "reply": "ご意見ありがとうございます。価格と内容量のバランスについて、いただいた声を商品開発の参考にさせていただきます。",
        "human_label": "pass",
        "human_note": "約束しすぎず、意見を受け止めている。",
    },
    {
        "id": 14,
        "review": "予約したケーキが受け取り時に用意できておらず、30分待たされました。",
        "reply": "ご予約商品をお待たせしてしまい、誠に申し訳ございません。予約管理の手順を改め、確実にご用意できる体制を整えます。",
        "human_label": "pass",
        "human_note": "謝罪 + 具体的な再発防止。",
    },
    {
        "id": 15,
        "review": "最高でした(笑)値段以外は。",
        "reply": "最高との評価、ありがとうございます!これからも最高のパンをお届けします!",
        "human_label": "fail",
        "human_note": "皮肉と価格への不満を完全に無視。ジャッジが割れやすい微妙ケース。",
    },
    {
        "id": 16,
        "review": "店内が少し暑く感じました。パンは美味しかったです。",
        "reply": "パンをお楽しみいただけたようで嬉しいです!店内温度についても貴重なご意見として、空調を調整いたします。ありがとうございます。",
        "human_label": "pass",
        "human_note": "良かった点と指摘の両方に対応。",
    },
]

df = pd.DataFrame(DATA)
print(f"件数: {len(df)}  (pass: {(df.human_label=='pass').sum()} / fail: {(df.human_label=='fail').sum()})")
display(df[["id", "review", "reply", "human_label"]])

## 3. ジャッジの定義

`make_judge()` でカスタムジャッジを作ります。テンプレート変数は `{{ inputs }}` / `{{ outputs }}` / `{{ expectations }}` / `{{ trace }}` の4つのみが使え、独自変数 (例: `{{ review }}`) はバリデーションエラーになります。ここでは `{{ inputs }}` にレビュー、`{{ outputs }}` に返信を渡します。

出力は `pass` / `fail` の2値。`feedback_value_type=Literal["pass", "fail"]` で型を固定します。

In [0]:
from mlflow.genai.judges import make_judge

judge = make_judge(
    name=JUDGE_NAME,
    instructions=(
        "あなたはベーカリーのカスタマーサポート品質を評価する審査員です。\n"
        "顧客レビュー {{ inputs }} に対するお店の返信 {{ outputs }} が適切かどうかを判定してください。\n\n"
        "以下をすべて満たす場合のみ pass、1つでも欠ける場合は fail としてください。\n"
        "- 感情の一致: レビューの感情 (肯定/否定/中立) に沿った返信になっている\n"
        "- 論点への対応: レビューで指摘された具体的な点に触れている\n"
        "- 苦情への謝罪: 否定的・問題を指摘するレビューには適切に謝罪・共感している\n"
        "- 事実の正確性: 裏付けのない割引・特典・安全性などを勝手に約束・断定していない\n"
        "- 安全性: 健康やアレルギーに関する根拠のない断定をしていない\n\n"
        "判定は pass または fail のいずれかを返してください。\n"
        "判定の理由 (rationale) は必ず日本語で記述してください。"
    ),
    feedback_value_type=Literal["pass", "fail"],
    model=JUDGE_MODEL,
)

print("ジャッジを作成しました:", judge.name)

## 4. ジャッジで全件を採点する

ジャッジは `judge(inputs=..., outputs=...)` で呼び出すと `mlflow.entities.Feedback` を返します。`.value` に判定 (pass/fail)、`.rationale` に判定理由が入ります。

In [0]:
judge_values = []
judge_rationales = []

for row in DATA:
    inputs = {"review": row["review"]}
    outputs = {"reply": row["reply"]}
    try:
        fb = judge(inputs=inputs, outputs=outputs)
        value = str(fb.value).strip().lower()
        # 念のためラベルを正規化
        value = "pass" if "pass" in value else ("fail" if "fail" in value else value)
        rationale = fb.rationale
    except Exception as e:
        value = "ERROR"
        rationale = f"{type(e).__name__}: {e}"
    judge_values.append(value)
    judge_rationales.append(rationale)
    print(f"id={row['id']:>2}  human={row['human_label']:<4}  judge={value:<4}")

df["judge_label"] = judge_values
df["judge_rationale"] = judge_rationales

# エラー行があれば除外して集計 (メタ評価は正常判定のみで行う)
n_error = (df["judge_label"] == "ERROR").sum()
if n_error:
    print(f"\n[警告] {n_error} 件でジャッジ呼び出しに失敗しました。JUDGE_MODEL のエンドポイント名を確認してください。")

eval_df = df[df["judge_label"].isin(LABELS)].copy()
display(eval_df[["id", "human_label", "judge_label"]])

## 5. 評価器を評価する (メタ評価)

ここが本題です。ジャッジの判定 `judge_label` と人間のゴールドラベル `human_label` の一致度を測ります。

- **一致率 (percent agreement)**: 単純に一致した割合。直感的だが、クラスが偏っていると高く出やすい
- **Cohen's kappa**: 偶然の一致を割り引いた指標。ジャッジの信頼性評価ではこちらが定番

kappaの目安 (Landis & Koch): 0.0以下=なし / 0.01-0.20=わずか / 0.21-0.40=低い / 0.41-0.60=中程度 / 0.61-0.80=かなり高い / 0.81-1.00=ほぼ完全。

In [0]:
def percent_agreement(y_human, y_judge):
    n = len(y_human)
    if n == 0:
        return float("nan")
    return sum(a == b for a, b in zip(y_human, y_judge)) / n


def cohen_kappa(y_human, y_judge, labels):
    """Cohen's kappa をライブラリ非依存で計算する。"""
    n = len(y_human)
    if n == 0:
        return float("nan")
    po = percent_agreement(y_human, y_judge)  # 観測一致率
    c_h = Counter(y_human)
    c_j = Counter(y_judge)
    pe = sum((c_h.get(l, 0) / n) * (c_j.get(l, 0) / n) for l in labels)  # 偶然一致率
    if pe == 1.0:
        return 1.0
    return (po - pe) / (1.0 - pe)


def interpret_kappa(k):
    if k < 0.01:
        return "なし (chance以下)"
    elif k <= 0.20:
        return "わずか (slight)"
    elif k <= 0.40:
        return "低い (fair)"
    elif k <= 0.60:
        return "中程度 (moderate)"
    elif k <= 0.80:
        return "かなり高い (substantial)"
    else:
        return "ほぼ完全 (almost perfect)"


y_human = eval_df["human_label"].tolist()
y_judge = eval_df["judge_label"].tolist()

pa = percent_agreement(y_human, y_judge)
kappa = cohen_kappa(y_human, y_judge, LABELS)

print(f"件数 (集計対象) : {len(eval_df)}")
print(f"一致率          : {pa:.3f}")
print(f"Cohen's kappa   : {kappa:.3f}  -> {interpret_kappa(kappa)}")

### 混同行列

行が人間ラベル、列がジャッジ判定です。対角以外がジャッジと人間のズレです。

In [0]:
confusion = pd.crosstab(
    eval_df["human_label"],
    eval_df["judge_label"],
    rownames=["human"],
    colnames=["judge"],
    dropna=False,
)
display(confusion)

## 6. 不一致の分析

メタ評価で本当に価値があるのは、この不一致ケースの深掘りです。ジャッジがどこで、なぜ人間とズレたのかを見ると、ジャッジのプロンプトをどう直すべきかが見えてきます。

In [0]:
disagree = eval_df[eval_df["human_label"] != eval_df["judge_label"]].copy()
print(f"不一致: {len(disagree)} / {len(eval_df)} 件\n")

for _, r in disagree.iterrows():
    print("=" * 80)
    print(f"id={r['id']}  human={r['human_label']}  judge={r['judge_label']}")
    print(f"[レビュー] {r['review']}")
    print(f"[返信]     {r['reply']}")
    print(f"[人間の意図] {r['human_note']}")
    print(f"[ジャッジの理由] {r['judge_rationale']}")
    print()

ここまでが「評価器を評価する」の最小構成です。セクション1-6だけで単体で動きます。

不一致ケースを見て、ジャッジのinstructionsを手で改善 -> 再採点 -> kappaを再計算、というループを回すのが基本の使い方です。次のセクションでは、それを人間フィードバックから自動で行うアラインメントを試します。

## 7. (任意) アラインメントでジャッジを人間に寄せる

MLflowのジャッジアラインメントは、人間フィードバックを使ってジャッジのプロンプトを自動最適化する機能です (デフォルトはSIMBAオプティマイザ)。

手順は、トレースを生成してジャッジ判定と人間ラベルの両方を記録し、`align()` で最適化するという流れです。要件はMLflow 3.4.0以上、トレースは最低10件 (推奨50件以上)、そして **人間フィードバックのnameがジャッジ名と完全一致していること** です。

このセクションは実行に時間がかかり、MLflowのバージョンによって挙動が変わることがあるため任意です。データが16件と少ないので、あくまで動作確認用と考えてください。

In [0]:
from mlflow.entities import AssessmentSource, AssessmentSourceType

# 各サンプルをトレース化し、ジャッジ判定 (LLM) と人間ラベル (HUMAN) を記録する
trace_ids = []
for row in DATA:
    with mlflow.start_span(name=f"reply_{row['id']}") as span:
        span.set_inputs({"review": row["review"]})
        span.set_outputs({"reply": row["reply"]})
    trace_ids.append(span.trace_id)

for tid, row in zip(trace_ids, DATA):
    trace = mlflow.get_trace(tid)
    inputs = {"review": row["review"]}
    outputs = {"reply": row["reply"]}

    # ジャッジ判定を記録 (nameはジャッジ名と一致させる)
    try:
        fb = judge(inputs=inputs, outputs=outputs)
        mlflow.log_feedback(
            trace_id=tid,
            name=JUDGE_NAME,
            value=fb.value,
            rationale=fb.rationale,
        )
    except Exception as e:
        print(f"id={row['id']} ジャッジ記録に失敗: {e}")

    # 人間フィードバックを記録 (nameはジャッジ名と一致させる)
    mlflow.log_feedback(
        trace_id=tid,
        name=JUDGE_NAME,
        value=row["human_label"],
        rationale=row["human_note"],
        source=AssessmentSource(
            source_type=AssessmentSourceType.HUMAN,
            source_id="gold_dataset",
        ),
    )

print(f"{len(trace_ids)} 件のトレースにジャッジ判定と人間ラベルを記録しました。")

In [0]:
# ジャッジと人間の両方のフィードバックが揃ったトレースでアラインメントを実行
from mlflow.genai.judges.optimizers import SIMBAAlignmentOptimizer

# 現在のノートブックエクスペリメント (アクティブなエクスペリメント) からトレースを取得
# experiment_ids は非推奨のため指定しない。トレースはこのノートブックで記録済み
traces_for_alignment = mlflow.search_traces(
    max_results=100,
    return_type="list",
)

valid_traces = []
for trace in traces_for_alignment:
    feedbacks = trace.search_assessments(name=JUDGE_NAME)
    has_judge = any(getattr(f.source, "source_type", None) in ("LLM_JUDGE", "CODE") for f in feedbacks)
    has_human = any(getattr(f.source, "source_type", None) == "HUMAN" for f in feedbacks)
    if has_judge and has_human:
        valid_traces.append(trace)

print(f"アラインメントに使えるトレース: {len(valid_traces)} 件")

aligned_judge = None
if len(valid_traces) >= 10:
    try:
        optimizer = SIMBAAlignmentOptimizer(model=JUDGE_MODEL)
        # シグネチャは align(traces, optimizer)。順序ミスを避けるためキーワードで渡す
        aligned_judge = judge.align(traces=valid_traces, optimizer=optimizer)
        print("アラインメント完了。")
    except Exception as e:
        print(f"アラインメントに失敗しました (バージョン差異の可能性): {type(e).__name__}: {e}")
else:
    print("トレースが不足しているためアラインメントをスキップします (最低10件必要)。")

### アラインメント前後のkappaを比較

アラインメント後のジャッジで同じデータを採点し直し、人間ラベルとのkappaが上がったかを確認します。

In [0]:
if aligned_judge is not None:
    aligned_values = []
    for row in DATA:
        try:
            fb = aligned_judge(inputs={"review": row["review"]}, outputs={"reply": row["reply"]})
            v = str(fb.value).strip().lower()
            v = "pass" if "pass" in v else ("fail" if "fail" in v else v)
        except Exception:
            v = "ERROR"
        aligned_values.append(v)

    aligned_df = df.copy()
    aligned_df["aligned_label"] = aligned_values
    aligned_df = aligned_df[aligned_df["aligned_label"].isin(LABELS)]

    ka = cohen_kappa(
        aligned_df["human_label"].tolist(),
        aligned_df["aligned_label"].tolist(),
        LABELS,
    )
    print(f"アラインメント前 kappa : {kappa:.3f}  -> {interpret_kappa(kappa)}")
    print(f"アラインメント後 kappa : {ka:.3f}  -> {interpret_kappa(ka)}")
else:
    print("アラインメント済みジャッジがないため比較をスキップします。")

## まとめ

- LLM-as-a-Judgeは便利だが、判定を鵜呑みにする前に「人間と合っているか」を測るメタ評価が必要
- 一致率だけでなく、偶然の一致を割り引く **Cohen's kappa** を併用する
- kappaの値そのものより、**不一致ケースの深掘り** がジャッジ改善の起点になる
- 人間フィードバックが溜まってきたら、`align()` によるアラインメントでジャッジを自動的に人間へ寄せられる

参考ドキュメント (日本語):
- カスタムジャッジ (make_judge): https://docs.databricks.com/aws/ja/mlflow3/genai/eval-monitor/custom-judge
- ジャッジを人間のフィードバックと整合させる (align): https://docs.databricks.com/aws/ja/mlflow3/genai/eval-monitor/align-judges